<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Data Wrangling Lab**


Estimated time needed: **45** minutes


In this lab, you will perform data wrangling tasks to prepare raw data for analysis. Data wrangling involves cleaning, transforming, and organizing data into a structured format suitable for analysis. This lab focuses on tasks like identifying inconsistencies, encoding categorical variables, and feature transformation.


## Objectives


After completing this lab, you will be able to:


- Identify and remove inconsistent data entries.

- Encode categorical variables for analysis.

- Handle missing values using multiple imputation strategies.

- Apply feature scaling and transformation techniques.


#### Intsall the required libraries


In [1]:
!pip install pandas
!pip install matplotlib

## Tasks


#### Step 1: Import the necessary module.


### 1. Load the Dataset


<h5>1.1 Import necessary libraries and load the dataset.</h5>


Ensure the dataset is loaded correctly by displaying the first few rows.


In [17]:
# Import necessary libraries
import pandas as pd
import numpy as np

# Load the Stack Overflow survey data
dataset_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/n01PQ9pSmiRX6520flujwQ/survey-data.csv"
df = pd.read_csv(dataset_url)

# Display the first few rows
print(df.head())


   ResponseId                      MainBranch                 Age  \
0           1  I am a developer by profession  Under 18 years old   
1           2  I am a developer by profession     35-44 years old   
2           3  I am a developer by profession     45-54 years old   
3           4           I am learning to code     18-24 years old   
4           5  I am a developer by profession     18-24 years old   

            Employment RemoteWork   Check  \
0  Employed, full-time     Remote  Apples   
1  Employed, full-time     Remote  Apples   
2  Employed, full-time     Remote  Apples   
3   Student, full-time        NaN  Apples   
4   Student, full-time        NaN  Apples   

                                    CodingActivities  \
0                                              Hobby   
1  Hobby;Contribute to open-source projects;Other...   
2  Hobby;Contribute to open-source projects;Other...   
3                                                NaN   
4                                 

#### 2. Explore the Dataset


<h5>2.1 Summarize the dataset by displaying the column data types, counts, and missing values.</h5>


In [7]:
# Write your code here



# Create summary
summary_df = pd.DataFrame({
    "Data Type": df.dtypes,
    "Non-Null Count": df.notnull().sum(),
    "Missing Values": df.isnull().sum()
})

# Optionally calculate missing value percentage
summary_df["Missing %"] = (summary_df["Missing Values"] / len(df)) * 100

# Display the summary
print(summary_df)


                    Data Type  Non-Null Count  Missing Values  Missing %
ResponseId              int64           65437               0   0.000000
MainBranch             object           65437               0   0.000000
Age                    object           65437               0   0.000000
Employment             object           65437               0   0.000000
RemoteWork             object           54806           10631  16.246160
...                       ...             ...             ...        ...
JobSatPoints_11       float64           29445           35992  55.002522
SurveyLength           object           56182            9255  14.143375
SurveyEase             object           56238            9199  14.057796
ConvertedCompYearly   float64           23435           42002  64.186928
JobSat                float64           29126           36311  55.490013

[114 rows x 4 columns]


<h5>2.2 Generate basic statistics for numerical columns.</h5>


In [9]:
# Write your code here

df.describe()

,ResponseId,CompTotal,WorkExp,JobSatPoints_1,JobSatPoints_4,JobSatPoints_5,JobSatPoints_6,JobSatPoints_7,JobSatPoints_8,JobSatPoints_9,JobSatPoints_10,JobSatPoints_11,ConvertedCompYearly,JobSat
count,65437.000000,3.374000e+04,29658.000000,29324.000000,29393.000000,29411.000000,29450.000000,29448.00000,29456.000000,29456.000000,29450.000000,29445.000000,2.343500e+04,29126.000000
mean,32719.000000,2.963841e+145,11.466957,18.581094,7.522140,10.060857,24.343232,22.96522,20.278165,16.169432,10.955713,9.953948,8.615529e+04,6.935041
std,18890.179119,5.444117e+147,9.168709,25.966221,18.422661,21.833836,27.089360,27.01774,26.108110,24.845032,22.906263,21.775652,1.867570e+05,2.088259
min,1.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,1.000000e+00,0.000000
25%,16360.000000,6.000000e+04,4.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,3.271200e+04,6.000000
50%,32719.000000,1.100000e+05,9.000000,10.000000,0.000000,0.000000,20.000000,15.00000,10.000000,5.000000,0.000000,0.000000,6.500000e+04,7.000000
75%,49078.000000,2.500000e+05,16.000000,22.000000,5.000000,10.000000,30.000000,30.00000,25.000000,20.000000,10.000000,10.000000,1.079715e+05,8.000000
max,65437.000000,1.000000e+150,50.000000,100.000000,100.000000,100.000000,100.000000,100.00000,100.000000,100.000000,100.000000,100.000000,1.625660e+07,10.000000


### 3. Identifying and Removing Inconsistencies


<h5>3.1 Identify inconsistent or irrelevant entries in specific columns (e.g., Country).</h5>


In [22]:
# Write your code here
import pandas as pd
from collections import Counter
import re

def clean_categorical_column(df, column_name, mapping_dict=None, irrelevant_entries=None, display_top=20):
    """
    Detect and clean inconsistent entries in a categorical column.
    
    Parameters:
    - df: pandas DataFrame
    - column_name: str, column to analyze
    - mapping_dict: dict, optional mapping of inconsistent entries to standard values
    - irrelevant_entries: list, optional entries to remove
    - display_top: int, number of top unique values to display
    
    Returns:
    - df_cleaned: DataFrame with a new column "{column_name}_Cleaned"
    """
    if column_name not in df.columns:
        print(f"Column {column_name} not found!")
        return df
    
    # Step 1: Display unique values and counts
    print(f"Top {display_top} unique values in '{column_name}':")
    print(df[column_name].value_counts(dropna=False).head(display_top))
    
    # Step 2: Apply mapping if provided
    df[column_name + "_Cleaned"] = df[column_name]
    if mapping_dict:
        df[column_name + "_Cleaned"] = df[column_name + "_Cleaned"].replace(mapping_dict)
    
    # Step 3: Remove irrelevant entries if provided
    if irrelevant_entries:
        df = df[~df[column_name + "_Cleaned"].isin(irrelevant_entries)]
    
    # Step 4: Strip whitespace and fix casing
    df[column_name + "_Cleaned"] = df[column_name + "_Cleaned"].astype(str).str.strip().str.title()
    
    # Step 5: Display cleaned top unique values
    print(f"\nTop {display_top} unique values after cleaning:")
    print(df[column_name + "_Cleaned"].value_counts(dropna=False).head(display_top))
    
    return df


<h5>3.2 Standardize entries in columns like Country or EdLevel by mapping inconsistent values to a consistent format.</h5>


In [44]:
df['EdLevel']

0                                Primary/elementary school
1             Bachelor’s degree (B.A., B.S., B.Eng., etc.)
2          Master’s degree (M.A., M.S., M.Eng., MBA, etc.)
3        Some college/university study without earning ...
4        Secondary school (e.g. American high school, G...
                               ...                        
65432         Bachelor’s degree (B.A., B.S., B.Eng., etc.)
65433                                              Unknown
65434         Bachelor’s degree (B.A., B.S., B.Eng., etc.)
65435    Secondary school (e.g. American high school, G...
65436                                              Unknown
Name: EdLevel, Length: 65437, dtype: object

In [42]:
!pip install thefuzz
import pandas as pd
from thefuzz import process

def standardize_categorical(df, column_name, reference_list=None, regex_patterns=None, fuzzy_threshold=80):
    """
    Standardizes a categorical column using:
    - Lowercase + strip
    - Optional regex replacements
    - Optional fuzzy matching to a reference list
    """
    df[column_name + "_Cleaned"] = df[column_name].astype(str).str.lower().str.strip()
    
    # Apply regex patterns
    if regex_patterns:
        for pattern, replacement in regex_patterns.items():
            df[column_name + "_Cleaned"] = df[column_name + "_Cleaned"].str.replace(pattern, replacement, regex=True)
    
    # Apply fuzzy matching
    if reference_list:
        def match_fuzzy(value):
            if value in reference_list:
                return value
            match, score = process.extractOne(value, reference_list)
            return match if score >= fuzzy_threshold else value
        df[column_name + "_Cleaned"] = df[column_name + "_Cleaned"].apply(match_fuzzy)
    
    # Capitalize cleaned entries
    df[column_name + "_Cleaned"] = df[column_name + "_Cleaned"].str.title()
    return df

# Example usage for Country
reference_countries = ["United States", "United Kingdom", "Canada", "India", "Australia"]
regex_country = {r"us|usa|u\.s\.a": "United States", r"uk|u\.k": "United Kingdom", r"canada": "Canada"}
df = standardize_categorical(df, "Country", reference_list=reference_countries, regex_patterns=regex_country)

# Example usage for EdLevel
regex_edlevel = {r"bachelor.*": "Bachelor's", r"master.*": "Master's", r"ph\.?d\.?": "PhD"}
df = standardize_categorical(df, "EdLevel", regex_patterns=regex_edlevel)


### 4. Encoding Categorical Variables


<h5>4.1 Encode the Employment column using one-hot encoding.</h5>


In [48]:
## Write your code here
import pandas as pd

# One-hot encode Employment column
employment_dummies = pd.get_dummies(df["Employment"], prefix="Employment")

# Concatenate the new one-hot columns with the original dataframe
df = pd.concat([df, employment_dummies], axis=1)

# Optional: drop the original column if no longer needed
# df.drop("Employment", axis=1, inplace=True)

# Check the result
df.head()


,ResponseId,MainBranch,Age,Employment,RemoteWork,Check,CodingActivities,EdLevel,LearnCode,LearnCodeOnline,...,"Employment_Student, full-time;Not employed, but looking for work;Not employed, and not looking for work;Student, part-time","Employment_Student, full-time;Not employed, but looking for work;Retired","Employment_Student, full-time;Not employed, but looking for work;Student, part-time","Employment_Student, full-time;Retired","Employment_Student, full-time;Student, part-time","Employment_Student, full-time;Student, part-time;Employed, part-time","Employment_Student, full-time;Student, part-time;Retired","Employment_Student, part-time","Employment_Student, part-time;Employed, part-time","Employment_Student, part-time;Retired"
0,1,I am a developer by profession,Under 18 years old,"Employed, full-time",Remote,Apples,Hobby,Primary/elementary school,Books / Physical media,NaN,...,False,False,False,False,False,False,False,False,False,False
1,2,I am a developer by profession,35-44 years old,"Employed, full-time",Remote,Apples,Hobby;Contribute to open-source projects;Other...,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Books / Physical media;Colleague;On the job tr...,Technical documentation;Blogs;Books;Written Tu...,...,False,False,False,False,False,False,False,False,False,False
2,3,I am a developer by profession,45-54 years old,"Employed, full-time",Remote,Apples,Hobby;Contribute to open-source projects;Other...,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Books / Physical media;Colleague;On the job tr...,Technical documentation;Blogs;Books;Written Tu...,...,False,False,False,False,False,False,False,False,False,False
3,4,I am learning to code,18-24 years old,"Student, full-time",NaN,Apples,NaN,Some college/university study without earning ...,"Other online resources (e.g., videos, blogs, f...",Stack Overflow;How-to videos;Interactive tutorial,...,False,False,False,False,False,False,False,False,False,False
4,5,I am a developer by profession,18-24 years old,"Student, full-time",NaN,Apples,NaN,"Secondary school (e.g. American high school, G...","Other online resources (e.g., videos, blogs, f...",Technical documentation;Blogs;Written Tutorial...,...,False,False,False,False,False,False,False,False,False,False


### 5. Handling Missing Values


<h5>5.1 Identify columns with the highest number of missing values.</h5>


In [50]:
## Write your code here

import pandas as pd

# Calculate missing values per column
missing_counts = df.isnull().sum()

# Sort columns by missing values in descending order
missing_counts_sorted = missing_counts.sort_values(ascending=False)

# Display top columns with the most missing values
print("Columns with the highest number of missing values:")
print(missing_counts_sorted.head(10))  # top 10 columns


Columns with the highest number of missing values:
AINextMuch less integrated       64289
AINextLess integrated            63082
AINextNo change                  52939
AINextMuch more integrated       51999
EmbeddedAdmired                  48704
EmbeddedWantToWorkWith           47837
EmbeddedHaveWorkedWith           43223
AIToolNot interested in Using    41023
AINextMore integrated            41009
Knowledge_9                      37802
dtype: int64


<h5>5.2 Impute missing values in numerical columns (e.g., `ConvertedCompYearly`) with the mean or median.</h5>


In [52]:
## Write your code here

df['ConvertedCompYearly'].fillna(df['ConvertedCompYearly'].median(), inplace=True)

# Option 2: Fill missing values with mean (good for symmetric data)
# df['ConvertedCompYearly'].fillna(df['ConvertedCompYearly'].mean(), inplace=True)

# Verify that there are no missing values left
print("Missing values in ConvertedCompYearly:", df['ConvertedCompYearly'].isnull().sum())

Missing values in ConvertedCompYearly: 0


C:\Users\praka\AppData\Local\Temp\ipykernel_21308\3772255163.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['ConvertedCompYearly'].fillna(df['ConvertedCompYearly'].median(), inplace=True)


<h5>5.3 Impute missing values in categorical columns (e.g., `RemoteWork`) with the most frequent value.</h5>


In [54]:
## Write your code here

mode_value = df['RemoteWork'].mode()[0]  # mode() returns a Series
df['RemoteWork'].fillna(mode_value, inplace=True)

# Verify that there are no missing values left
print("Missing values in RemoteWork:", df['RemoteWork'].isnull().sum())

Missing values in RemoteWork: 0


C:\Users\praka\AppData\Local\Temp\ipykernel_21308\1555393774.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['RemoteWork'].fillna(mode_value, inplace=True)


### 6. Feature Scaling and Transformation


<h5>6.1 Apply Min-Max Scaling to normalize the `ConvertedCompYearly` column.</h5>


In [56]:
## Write your code here

import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# Initialize the scaler
scaler = MinMaxScaler()

# Reshape the column to 2D (required by sklearn)
converted_comp = df[['ConvertedCompYearly']]

# Apply Min-Max scaling
df['ConvertedCompYearly_MinMax'] = scaler.fit_transform(converted_comp)

# Check the result
print(df[['ConvertedCompYearly', 'ConvertedCompYearly_MinMax']].head())


   ConvertedCompYearly  ConvertedCompYearly_MinMax
0              65000.0                    0.003998
1              65000.0                    0.003998
2              65000.0                    0.003998
3              65000.0                    0.003998
4              65000.0                    0.003998


<h5>6.2 Log-transform the ConvertedCompYearly column to reduce skewness.</h5>


In [58]:
## Write your code here

import pandas as pd
import numpy as np

# Replace or filter negative/zero values if necessary
# Add a small constant (e.g., 1) to avoid log(0)
df['ConvertedCompYearly_Log'] = np.log1p(df['ConvertedCompYearly'])

# Check the result
print(df[['ConvertedCompYearly', 'ConvertedCompYearly_Log']].head())




   ConvertedCompYearly  ConvertedCompYearly_Log
0              65000.0                11.082158
1              65000.0                11.082158
2              65000.0                11.082158
3              65000.0                11.082158
4              65000.0                11.082158


### 7. Feature Engineering


<h5>7.1 Create a new column `ExperienceLevel` based on the `YearsCodePro` column:</h5>


In [64]:
## Write your code here

import pandas as pd

# Convert YearsCodePro to numeric, coercing errors to NaN
df['YearsCodePro'] = pd.to_numeric(df['YearsCodePro'], errors='coerce')

# Optionally fill NaN with 0 or some default value
df['YearsCodePro'].fillna(0, inplace=True)

# Define a function to categorize experience
def categorize_experience(years):
    if years < 2:
        return "Junior"
    elif 2 <= years < 5:
        return "Intermediate"
    elif 5 <= years < 10:
        return "Senior"
    else:
        return "Expert"

# Apply the function to create the new column
df['ExperienceLevel'] = df['YearsCodePro'].apply(categorize_experience)

# Check the result
print(df[['YearsCodePro', 'ExperienceLevel']].head(10))


   YearsCodePro ExperienceLevel
0           0.0          Junior
1          17.0          Expert
2          27.0          Expert
3           0.0          Junior
4           0.0          Junior
5           0.0          Junior
6           7.0          Senior
7           0.0          Junior
8           0.0          Junior
9          11.0          Expert


C:\Users\praka\AppData\Local\Temp\ipykernel_21308\3492886504.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['YearsCodePro'].fillna(0, inplace=True)


### Summary


In this lab, you:

- Explored the dataset to identify inconsistencies and missing values.

- Encoded categorical variables for analysis.

- Handled missing values using imputation techniques.

- Normalized and transformed numerical data to prepare it for analysis.

- Engineered a new feature to enhance data interpretation.


Copyright © IBM Corporation. All rights reserved.
